# Part 12 — Deployment & Capstone

**PyTimeSeries user guide** · *The deployment loop, plus the grand statistical-vs-ML-vs-deep benchmark.*

> Uses the `pytimeseries` package on the shared commodity-price dataset.

In [1]:
import warnings; warnings.simplefilter("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import pytimeseries as pt
plt.rcParams.update({"figure.figsize": (10, 3.8), "axes.grid": True, "grid.alpha": .3,
                     "axes.spines.top": False, "axes.spines.right": False})

# Shared dataset: 356 months of global commodity prices (1992-2021)
raw = pd.read_csv("monthly_price.csv")
dates = pd.to_datetime(raw["YEAR"].astype(str) + "-" + raw["MONTH"], format="%Y-%b")
df = raw.drop(columns=["ID", "YEAR", "MONTH", "TIME"]); df.index = dates
df.index.name = "date"; df = df.asfreq("MS")
y = df["Food_Price_Index"]
print("pytimeseries", pt.__version__, "| series:", list(df.columns))

pytimeseries 0.5.0 | series: ['Urea_Price', 'MP_Price', 'TSP_Price', 'Maize_Price', 'Rice_Price', 'Wheat_Price', 'Food_Price_Index', 'Energy_Price_Index', 'VIX_Index']


The capstone: one walk-forward benchmark across three families, then the deployment loop (persist, monitor drift, retrain, serve).

In [2]:
cv = pt.ExpandingWindow(h=12, n_splits=5)
board = pd.concat([
    pt.evaluate(pt.ETS(seasonal='add', seasonal_periods=12), y, cv=cv, metrics=('MASE',), sp=12).summary.loc[['ETS']],
    pt.evaluate(pt.make_reduction(pt.ml.LightGBM(), lags=12), y, cv=cv, metrics=('MASE',), sp=12).summary.loc[['RecursiveReductionForecaster']],
    pt.evaluate(pt.deep.LSTMForecaster(lags=12, epochs=60), y, cv=cv, metrics=('MASE',), sp=12).summary.loc[['LSTMForecaster']],
]).sort_values('MASE')
board

metric,MASE
model,
ETS,0.6858
LSTMForecaster,0.7982
RecursiveReductionForecaster,1.1215


In [3]:
# deployment loop
model = pt.ETS(seasonal='add', seasonal_periods=12).fit(y)
pt.deploy.save(model, 'model.joblib', metadata={'owner': 'zia207'})
print('reload identical:', bool((pt.deploy.load('model.joblib').predict(6).round(6) == model.predict(6).round(6)).all()))
mon = pt.deploy.DriftMonitor(threshold=2.0).fit(y[:120])
pol = pt.deploy.RetrainPolicy(every=12, drift_monitor=mon)
print('retrain @ step 12?', pol.should_retrain(step=12, window=y[-12:]))

reload identical: True
retrain @ step 12? True


**Takeaway.** The same `BaseForecaster` contract that carried us from `Naive` to `PatchTST` also persists, monitors, and serves — the full lifecycle in one library.

*End of the pytimeseries user guide.*